### Downloading PDFs

In [ ]:
from selenium.webdriver.chrome.options import Options
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
import time
import os
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

BASE_DIR = os.path.join("/Users/aryankhurana/Electricity-Forecasting-Indian-States")
DOWNLOAD_DIR = os.path.join(BASE_DIR, "artifacts", "downloaded_pdfs")

URL = "https://grid-india.in/en/reports/weekly-report"

# Ensure download directory exists
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# === Setup Selenium with Brave ===
options = Options()
options.binary_location = "C:\\Program Files\\BraveSoftware\\Brave-Browser\\Application\\brave.exe"
service = Service(executable_path=r"/Users/aryankhurana/Downloads/Softwares/chromedriver-mac-arm64/chromedriver")
driver = webdriver.Chrome(service=service, options=options)

driver.get(URL)
wait = WebDriverWait(driver, 10)

# === STEP 1: Select "ALL" from dropdown ===
try:
    dropdown_control = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "div.my-select__control")
    ))
    dropdown_control.click()
    time.sleep(1)

    # Wait for dropdown menu to appear
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div[class*='my-select__menu']")
    ))

    # Find and click ALL option
    all_option = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//div[contains(@class, 'my-select__option') and text()='ALL']")
    ))
    all_option.click()
    time.sleep(2)

    print("Selected ALL option")
    
except Exception as e:
    print("Dropdown selection failed:", e)
    driver.quit()
    exit()

# === STEP 2: Scrape PDF links with pagination ===
all_pdf_links = set()
page_number = 1

while True:
    print(f"Scraping Page {page_number}...")
    time.sleep(2)

    try:
        # Wait for PDF links to load
        wait.until(EC.presence_of_all_elements_located((By.XPATH, "//a[contains(@href, '.pdf')]")))

        pdf_links = driver.find_elements(By.XPATH, "//a[contains(@href, '.pdf')]")
        for link in pdf_links:
            driver.execute_script("window.scrollBy(0, 52);")
            time.sleep(1)

            href = link.get_attribute("href")
            if href and href.endswith(".pdf"):
                all_pdf_links.add(href)

        try:
            next_button = driver.find_element(By.XPATH, "//button[@aria-label='Next Page']")
            if next_button.get_attribute("disabled") is not None:
                print("Reached the last page.")
                break
            next_button.click()
            time.sleep(2)
            page_number += 1
        except Exception as e:
            print("Error navigating to next page:", e)
            break
        # Scroll to the very top of the page
        driver.execute_script("window.scrollTo(0, 100);")
        time.sleep(1)

    except Exception as e:
        print("Error during pagination:", e)
        break
driver.quit()
try:
    print(f"Total PDFs found: {len(all_pdf_links)}. Downloading...")
    for i, link in enumerate(all_pdf_links, 1):
        filename = os.path.join(DOWNLOAD_DIR, f"report_{i:03d}.pdf")
        try:
            response = requests.get(link)
            with open(filename, "wb") as f:
                f.write(response.content)
            print(f"Downloaded: {filename}")
        except Exception as e:
            print(f"Failed to download {link}: {e}")

    print("✅ All downloads complete.")
except Exception as e:
    print(f"Error during download: {e}")



: 

### Extracting Tables from PDFs

In [ ]:
import os
import tabula
import PyPDF2
import pandas as pd

# Folder paths
pdf_dir = os.path.join(BASE_DIR, "artifacts", "downloaded_pdfs")
csv_dir = os.path.join(BASE_DIR, "artifacts", "downloaded_csvs", "csv_files")

os.makedirs(csv_dir, exist_ok=True)

# List all PDF files in the directory
pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith(".pdf")]

# Process each PDF
for idx, file_path in enumerate(pdf_files, 1):
    try:
        print(f"Processing {file_path}...")

        with open(file_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            page_found = False

            for i in range(1, len(reader.pages)):
                text = reader.pages[i].extract_text()
                if text and 'Energy Consumption in States (MUs)' in text:
                    page_found = True
                    required_page = i + 1  # tabula is 1-based
                    break

        if not page_found:
            print(f"❌ Table not found in {file_path}")
            continue

        # Extract table from the page using tabula
        tables = tabula.read_pdf(file_path, pages=required_page, multiple_tables=True)

        if tables:
            df = tables[0]
            # Drop the last row if it is total or NaN
            df = df.dropna(how='all')
            if 'ALL INDIA TOTAL' in df.iloc[-1].astype(str).values:
                df = df.iloc[:-1]

            csv_filename = os.path.splitext(os.path.basename(file_path))[0] + ".csv"
            csv_path = os.path.join(csv_dir, csv_filename)
            df.to_csv(csv_path, index=False)
            print(f"✅ Saved to: {csv_path}")
        else:
            print(f"⚠️ No tables extracted from page {required_page} in {file_path}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")


Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_001.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_001.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_002.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_002.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_003.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_003.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_004.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_004.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_005.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_005.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs

Got stderr: Jul 21, 2025 11:35:49 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider loadDiskCache
Jul 21, 2025 11:35:49 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider <init>
Jul 21, 2025 11:35:50 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider <init>



✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_131.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_132.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_132.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_133.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_133.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_134.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_134.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_135.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_135.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_136.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csv

Got stderr: Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:36:22 PM org.apache.pdfbox.pdmodel.f

✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_143.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_144.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_144.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_145.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_145.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_146.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_146.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_147.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_147.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_148.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csv

Got stderr: Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 21, 2025 11:41:30 PM org.apache.pdfbox.pdmodel.f

✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_279.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_280.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_280.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_281.pdf...
❌ Table not found in E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_281.pdf
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_282.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_282.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_283.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs\csv_files\report_283.csv
Processing E:\Electricity Demand Predicition\artifacts\downloaded_pdfs\report_284.pdf...
✅ Saved to: E:\Electricity Demand Predicition\artifacts\downloaded_csvs

In [5]:
!pip install JPype1


In [8]:
import tabula

tabula.environment_info()


Python version:
    3.10.18 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:08:55) [MSC v.1929 64 bit (AMD64)]
Java version:
    java version "24.0.2" 2025-07-15
Java(TM) SE Runtime Environment (build 24.0.2+12-54)
Java HotSpot(TM) 64-Bit Server VM (build 24.0.2+12-54, mixed mode, sharing)
tabula-py version: 2.10.0
platform: Windows-10-10.0.26100-SP0
uname:
    uname_result(system='Windows', node='LAPTOP-HP1IG7O1', release='10', version='10.0.26100', machine='AMD64')
linux_distribution: ('', '', '')
mac_ver: ('', ('', '', ''), '')


In [7]:
import os

# Add Java path for tabula to work in Anaconda
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Java\jdk-24\bin"

import tabula

tabula.environment_info()


Python version:
    3.10.18 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:08:55) [MSC v.1929 64 bit (AMD64)]
Java version:
    java version "24.0.2" 2025-07-15
Java(TM) SE Runtime Environment (build 24.0.2+12-54)
Java HotSpot(TM) 64-Bit Server VM (build 24.0.2+12-54, mixed mode, sharing)
tabula-py version: 2.10.0
platform: Windows-10-10.0.26100-SP0
uname:
    uname_result(system='Windows', node='LAPTOP-HP1IG7O1', release='10', version='10.0.26100', machine='AMD64')
linux_distribution: ('', '', '')
mac_ver: ('', ('', '', ''), '')


### Cleaning

### Combining Tables

In [ ]:
import pandas as pd
import numpy as np